In [1]:
!pip -q install pandas numpy scikit-learn requests mygene torch torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00


# Imports And Configuration

In [2]:
import gzip, json, random, re, time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import mygene
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv

CONTENT = Path("/content")
OUTPUT_DIR = CONTENT / "spaceflight_drug_model_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

DATASET_FILES = {
    "GLDS-48": CONTENT / "GLDS-48_rna_seq_differential_expression_GLbulkRNAseq.csv",
    "GLDS-246": CONTENT / "GLDS-246_rna_seq_differential_expression_GLbulkRNAseq.csv",
    "GLDS-288": CONTENT / "GLDS-288_rna_seq_differential_expression_GLbulkRNAseq.csv",
    "GLDS-289": CONTENT / "GLDS-289_rna_seq_differential_expression_GLbulkRNAseq.csv",
    "GLDS-245": CONTENT / "GLDS-245_rna_seq_differential_expression_GLbulkRNAseq.csv",
    "GLDS-244": CONTENT / "GLDS-244_rna_seq_differential_expression_GLbulkRNAseq.csv",
    "GLDS-4": CONTENT / "GLDS-4_array_differential_expression.csv",
}

CTD_URL = "https://ctdbase.org/reports/CTD_chem_gene_ixns.csv.gz"
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"

SPACEFLIGHT_TERMS = ("space flight", "spaceflight", "on iss", "flight", "flt", "hindlimb", "unloading", "hu")
CONTROL_TERMS = ("ground control", "basal control", "control", "gc", "vivarium", "1g", "sham")
BAD_CHEMICAL_TERMS = ("bisphenol", "benzo(a)pyrene", "benzopyrene", "amyloid beta", "diesel", "dioxin", "pcb", "arsenic", "cadmium", "lead", "mercury", "asbestos", "lipopolysaccharide", "formaldehyde")

@dataclass
class Config:
    fdr_threshold: float = 0.05
    raw_p_fallback_threshold: float = 0.01
    min_abs_log2fc: float = 1.0
    min_dataset_support: int = 2
    min_ctd_pubmed_count: int = 5
    max_chembl_genes: int = 120
    hidden_channels: int = 64
    epochs: int = 120
    lr: float = 0.005
    weight_decay: float = 1e-4
    margin: float = 1.0
    hard_negative_degree_tolerance: int = 5
    seed: int = 42

CONFIG = Config()
random.seed(CONFIG.seed)
np.random.seed(CONFIG.seed)
torch.manual_seed(CONFIG.seed)

for name, path in DATASET_FILES.items():
    print(name, "FOUND" if path.exists() else "MISSING", path)

GLDS-48 FOUND /content/GLDS-48_rna_seq_differential_expression_GLbulkRNAseq.csv
GLDS-246 FOUND /content/GLDS-246_rna_seq_differential_expression_GLbulkRNAseq.csv
GLDS-288 FOUND /content/GLDS-288_rna_seq_differential_expression_GLbulkRNAseq.csv
GLDS-289 FOUND /content/GLDS-289_rna_seq_differential_expression_GLbulkRNAseq.csv
GLDS-245 FOUND /content/GLDS-245_rna_seq_differential_expression_GLbulkRNAseq.csv
GLDS-244 FOUND /content/GLDS-244_rna_seq_differential_expression_GLbulkRNAseq.csv
GLDS-4 FOUND /content/GLDS-4_array_differential_expression.csv


# DEG Extraction Functions

In [3]:
def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    p = np.where(np.isfinite(p), p, 1.0)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = np.empty(n)
    running = 1.0

    for rank in range(n, 0, -1):
        i = rank - 1
        running = min(running, ranked[i] * n / rank)
        adjusted[order[i]] = min(1.0, running)

    return adjusted

def clean_gene(x):
    if x is None or pd.isna(x):
        return None
    x = str(x).strip()
    return None if x.upper() in {"", "NA", "NAN", "NONE"} else x

def split_contrast(col):
    base = re.sub(r"^(Log2fc|P\.value|P\.adj|padj|FDR|qvalue)_", "", col, flags=re.I)
    m = re.match(r"^\((.*)\)v\((.*)\)$", base)
    return (base, m.group(1), m.group(2)) if m else (base, None, None)

def get_duration(text):
    m = re.search(r"~?\s*(\d+)\s*day", text.lower())
    return int(m.group(1)) if m else None

def has_any(text, terms):
    text = text.lower()
    return any(t in text for t in terms)

def contrast_orientation(fc_col):
    """
    Returns:
      +1 if column is Treatment v Control
      -1 if column is Control v Treatment
      None if invalid
    """
    _, left, right = split_contrast(fc_col)
    if not left or not right:
        return None

    left_treat = has_any(left, SPACEFLIGHT_TERMS)
    right_treat = has_any(right, SPACEFLIGHT_TERMS)
    left_ctrl = has_any(left, CONTROL_TERMS)
    right_ctrl = has_any(right, CONTROL_TERMS)

    dl, dr = get_duration(left), get_duration(right)
    if dl is not None and dr is not None and dl != dr:
        return None

    if left_treat and right_ctrl:
        return +1

    if left_ctrl and right_treat:
        return -1

    return None

def valid_contrast(fc_col):
    return contrast_orientation(fc_col) is not None

def matching_p_col(columns, fc_col):
    base, _, _ = split_contrast(fc_col)
    key = re.sub(r"\s+", "", base.lower())

    for c in columns:
        if re.match(r"^(P\.value|P\.adj|padj|FDR|qvalue)_", c, flags=re.I):
            pbase, _, _ = split_contrast(c)
            if re.sub(r"\s+", "", pbase.lower()) == key:
                return c

    return None

print("Direction-aware DEG helper functions ready.")

Direction-aware DEG helper functions ready.


# Run DEG Extraction And Consensus

In [4]:
def load_dataset_degs(dataset_id, path, config):
    print(f"\nLoading {dataset_id}: {path.name}")
    df = pd.read_csv(path, low_memory=False)

    fc_cols = [
        c for c in df.columns
        if c.lower().startswith("log2fc_") and valid_contrast(c)
    ]

    print("Strict contrasts found:", len(fc_cols))

    pieces = []

    for fc in fc_cols:
        pc = matching_p_col(df.columns, fc)
        orient = contrast_orientation(fc)

        if pc is None or orient is None:
            continue

        raw_fc = pd.to_numeric(df[fc], errors="coerce")

        tmp = pd.DataFrame({
            "dataset": dataset_id,
            "gene_mouse": df["SYMBOL"].map(clean_gene),
            "log2fc_raw": raw_fc,
            "log2fc": raw_fc * orient,
            "pvalue": pd.to_numeric(df[pc], errors="coerce"),
            "contrast": fc,
            "orientation": orient,
        }).dropna()

        tmp["qvalue"] = bh_fdr(tmp["pvalue"])

        keep = tmp[
            (tmp.qvalue < config.fdr_threshold)
            & (tmp.log2fc.abs() >= config.min_abs_log2fc)
        ].copy()

        keep["used_raw_p_fallback"] = False

        if keep.empty:
            keep = tmp[
                (tmp.pvalue < config.raw_p_fallback_threshold)
                & (tmp.log2fc.abs() >= config.min_abs_log2fc)
            ].copy()
            keep["used_raw_p_fallback"] = True

        pieces.append(keep)

    out = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()
    print("Filtered DEG rows:", len(out))
    return out

all_degs = []

for dataset_id, path in DATASET_FILES.items():
    if not path.exists():
        raise FileNotFoundError(path)
    all_degs.append(load_dataset_degs(dataset_id, path, CONFIG))

degs = pd.concat(all_degs, ignore_index=True)

consensus_mouse = (
    degs.groupby("gene_mouse")
    .agg(
        dataset_support=("dataset", lambda x: len(set(x))),
        datasets=("dataset", lambda x: ",".join(sorted(set(x)))),
        mean_log2fc=("log2fc", "mean"),
        median_log2fc=("log2fc", "median"),
        mean_abs_log2fc=("log2fc", lambda x: float(np.mean(np.abs(x)))),
        direction_score=("log2fc", lambda x: float(np.sign(x).mean())),
        min_qvalue=("qvalue", "min"),
        best_pvalue=("pvalue", "min"),
        contrast_count=("contrast", "nunique"),
    )
    .reset_index()
)

consensus_mouse["consensus_weight"] = (
    consensus_mouse.dataset_support
    * consensus_mouse.mean_abs_log2fc
    * consensus_mouse.direction_score.abs()
)

consensus_mouse = consensus_mouse[
    consensus_mouse.dataset_support >= CONFIG.min_dataset_support
]

consensus_mouse = consensus_mouse.sort_values(
    ["dataset_support", "consensus_weight"],
    ascending=False,
).reset_index(drop=True)

print("\nTotal DEG rows:", len(degs))
print("Consensus mouse genes:", len(consensus_mouse))
display(consensus_mouse.head(30))


Loading GLDS-48: GLDS-48_rna_seq_differential_expression_GLbulkRNAseq.csv
Strict contrasts found: 8
Filtered DEG rows: 12394

Loading GLDS-246: GLDS-246_rna_seq_differential_expression_GLbulkRNAseq.csv
Strict contrasts found: 4
Filtered DEG rows: 114

Loading GLDS-288: GLDS-288_rna_seq_differential_expression_GLbulkRNAseq.csv
Strict contrasts found: 6
Filtered DEG rows: 224

Loading GLDS-289: GLDS-289_rna_seq_differential_expression_GLbulkRNAseq.csv
Strict contrasts found: 28
Filtered DEG rows: 10660

Loading GLDS-245: GLDS-245_rna_seq_differential_expression_GLbulkRNAseq.csv
Strict contrasts found: 4
Filtered DEG rows: 958

Loading GLDS-244: GLDS-244_rna_seq_differential_expression_GLbulkRNAseq.csv
Strict contrasts found: 4
Filtered DEG rows: 5998

Loading GLDS-4: GLDS-4_array_differential_expression.csv
Strict contrasts found: 2
Filtered DEG rows: 6

Total DEG rows: 30354
Consensus mouse genes: 1658


,gene_mouse,dataset_support,datasets,mean_log2fc,median_log2fc,mean_abs_log2fc,direction_score,min_qvalue,best_pvalue,contrast_count,consensus_weight
0,Npas2,4,"GLDS-245,GLDS-246,GLDS-289,GLDS-48",2.660256,2.505043,2.660256,1.000000,3.433588e-16,3.818704e-20,8,10.641023
1,Cyp2a5,4,"GLDS-244,GLDS-245,GLDS-289,GLDS-48",0.063468,0.000000,1.959895,0.000000,3.942918e-10,3.223093e-12,8,0.000000
2,Gm15500,3,"GLDS-288,GLDS-289,GLDS-48",-9.433882,-13.138216,16.241965,-0.500000,2.178289e-27,1.202677e-31,20,24.362948
3,Gm23238,3,"GLDS-244,GLDS-245,GLDS-289",-4.278804,-4.991243,4.278804,-1.000000,3.900973e-03,1.506550e-04,10,12.836411
4,Slc34a2,3,"GLDS-244,GLDS-245,GLDS-48",3.371168,2.956145,3.371168,1.000000,8.714115e-07,2.979030e-09,8,10.113503
5,Col5a3,3,"GLDS-244,GLDS-245,GLDS-48",2.672380,2.474566,2.672380,1.000000,6.540812e-07,2.032781e-09,4,8.017139
6,Aqp8,3,"GLDS-244,GLDS-245,GLDS-48",2.638912,1.291768,2.638912,1.000000,3.564103e-07,3.601492e-10,4,7.916736
7,Igkv3-2,3,"GLDS-244,GLDS-289,GLDS-48",1.722424,3.418134,4.285246,0.600000,2.676638e-03,7.871952e-05,10,7.713443
8,Cacna1h,3,"GLDS-244,GLDS-289,GLDS-48",2.489354,2.243153,2.745720,0.857143,4.134197e-06,3.583641e-08,14,7.060424
9,Dmkn,3,"GLDS-244,GLDS-289,GLDS-48",-1.708257,1.729090,11.756778,0.200000,2.421567e-10,4.200443e-12,10,7.054067


# Robust Mouse To Human Ortholog Mapping

In [5]:
def same_symbol_human_fallback(mouse_gene):
    mg = mygene.MyGeneInfo()

    hits = mg.query(
        mouse_gene,
        scopes="symbol,alias",
        species="human",
        fields="symbol,taxid,type_of_gene,name",
        size=10,
    )

    for h in hits.get("hits", []):
        if str(h.get("taxid")) == "9606" and h.get("symbol"):
            return h["symbol"].upper()

    return None

def extract_mouse_ensembl_ids(hit):
    ens = hit.get("ensembl")
    ids = []

    if isinstance(ens, dict) and ens.get("gene"):
        ids.append(ens["gene"])

    elif isinstance(ens, list):
        for item in ens:
            if isinstance(item, dict) and item.get("gene"):
                ids.append(item["gene"])

    return list(dict.fromkeys(ids))

def ensembl_symbol_to_ids(mouse_gene):
    urls = [
        f"https://rest.ensembl.org/xrefs/symbol/mouse/{mouse_gene}",
        f"https://rest.ensembl.org/xrefs/symbol/mus_musculus/{mouse_gene}",
    ]

    ids = []

    for url in urls:
        try:
            r = requests.get(
                url,
                headers={"Content-Type": "application/json"},
                timeout=30,
            )
            if r.status_code != 200:
                continue

            for item in r.json():
                if item.get("type") == "gene" and item.get("id", "").startswith("ENSMUSG"):
                    ids.append(item["id"])

        except Exception:
            pass

    return list(dict.fromkeys(ids))

def ensembl_mouse_to_human(mouse_ensembl):
    urls = [
        f"https://rest.ensembl.org/homology/id/mouse/{mouse_ensembl}",
        f"https://rest.ensembl.org/homology/id/mus_musculus/{mouse_ensembl}",
    ]

    last_error = None

    for url in urls:
        try:
            r = requests.get(
                url,
                params={
                    "target_species": "human",
                    "type": "orthologues",
                },
                headers={"Content-Type": "application/json"},
                timeout=30,
            )

            if r.status_code == 404:
                last_error = f"404 for {url}"
                continue

            r.raise_for_status()
            payload = r.json()

            hits = []

            for rec in payload.get("data", []):
                for h in rec.get("homologies", []):
                    target = h.get("target", {})

                    if target.get("species") != "homo_sapiens":
                        continue

                    hits.append({
                        "human_ensembl": target.get("id"),
                        "human_symbol": target.get("display_id"),
                        "orthology_type": h.get("type"),
                        "perc_id": target.get("perc_id", 0),
                    })

            hits = sorted(
                hits,
                key=lambda x: (
                    "ortholog_one2one" in str(x.get("orthology_type")),
                    float(x.get("perc_id") or 0),
                ),
                reverse=True,
            )

            if hits:
                return hits

        except Exception as e:
            last_error = str(e)

    raise RuntimeError(last_error or f"No Ensembl homology result for {mouse_ensembl}")

def resolve_human_symbols(ensembl_ids):
    if not ensembl_ids:
        return {}

    mg = mygene.MyGeneInfo()

    hits = mg.querymany(
        list(ensembl_ids),
        scopes="ensembl.gene",
        species="human",
        fields="symbol,taxid",
        as_dataframe=False,
        verbose=False,
    )

    out = {}

    for h in hits:
        if h.get("query") and h.get("symbol") and str(h.get("taxid")) == "9606":
            out[h["query"]] = h["symbol"].upper()

    return out

def map_mouse_to_human(mouse_genes):
    print("Mapping mouse genes to human orthologs...")

    mg = mygene.MyGeneInfo()

    mouse_hits = mg.querymany(
        list(mouse_genes),
        scopes="symbol",
        species="mouse",
        fields="symbol,ensembl,taxid",
        as_dataframe=False,
        verbose=False,
    )

    mouse_to_ens = {
        h.get("query"): extract_mouse_ensembl_ids(h)
        for h in mouse_hits
        if h.get("query")
    }

    rows = []
    unresolved = []

    for gene in mouse_genes:
        mapped = False

        # First: validate direct human same-symbol ortholog.
        # For genes like Npas2 -> NPAS2, S100a8 -> S100A8, Cpt1a -> CPT1A,
        # this is often the cleanest route.
        same_symbol = same_symbol_human_fallback(gene)

        if same_symbol:
            rows.append({
                "mouse_gene": gene,
                "mouse_ensembl": None,
                "human_gene": same_symbol,
                "human_ensembl": None,
                "orthology_type": "same_symbol_human_validated",
                "perc_id": None,
                "source": "mygene_human_symbol_validation",
            })
            mapped = True

        # Second: try Ensembl homology with correct species URL.
        if not mapped:
            candidate_ens_ids = list(mouse_to_ens.get(gene, []))

            # If MyGene gives an old/dead Ensembl ID, ask Ensembl for current IDs by symbol.
            candidate_ens_ids += ensembl_symbol_to_ids(gene)
            candidate_ens_ids = list(dict.fromkeys(candidate_ens_ids))

            for ens in candidate_ens_ids:
                try:
                    human_hits = ensembl_mouse_to_human(ens)
                    human_ensembl_ids = [
                        x["human_ensembl"]
                        for x in human_hits
                        if x.get("human_ensembl")
                    ]

                    symbol_map = resolve_human_symbols(human_ensembl_ids)

                    for x in human_hits:
                        human_ensembl = x.get("human_ensembl")
                        human_symbol = symbol_map.get(human_ensembl) or x.get("human_symbol")

                        if human_symbol:
                            rows.append({
                                "mouse_gene": gene,
                                "mouse_ensembl": ens,
                                "human_gene": str(human_symbol).upper(),
                                "human_ensembl": human_ensembl,
                                "orthology_type": x.get("orthology_type"),
                                "perc_id": x.get("perc_id"),
                                "source": "ensembl_homology",
                            })
                            mapped = True
                            break

                    if mapped:
                        break

                except Exception as e:
                    print("Skipped", gene, ens, e)

                time.sleep(0.1)

        if not mapped:
            unresolved.append(gene)

    orthologs = pd.DataFrame(rows).drop_duplicates(["mouse_gene", "human_gene"])

    print("Mapped ortholog rows:", len(orthologs))
    print("Unresolved:", unresolved)

    if orthologs.empty:
        raise ValueError("No orthologs mapped.")

    return orthologs

orthologs = map_mouse_to_human(consensus_mouse.gene_mouse.tolist())

consensus_human = consensus_mouse.merge(
    orthologs,
    left_on="gene_mouse",
    right_on="mouse_gene",
    how="inner",
)

consensus_human["human_gene"] = consensus_human["human_gene"].str.upper()

# Remove genes with zero direction-consensus weight.
# These had conflicting direction across contrasts and should not drive disease edges.
consensus_human = consensus_human[consensus_human["consensus_weight"] > 0].copy()

human_genes = set(consensus_human.human_gene)

print("Human consensus genes:", len(human_genes))
display(consensus_human.head(30))

Mapping mouse genes to human orthologs...
Skipped Gm15500 ENSMUSG00000086583 No Ensembl homology result for ENSMUSG00000086583
Skipped Gm23238 ENSMUSG00000064941 No Ensembl homology result for ENSMUSG00000064941
Skipped Gm23238 ENSMSIG00000010995 No Ensembl homology result for ENSMSIG00000010995
Skipped BC049987 ENSMUSG00000110755 No Ensembl homology result for ENSMUSG00000110755
Skipped Ugt2b34 ENSMUSG00000029260 No Ensembl homology result for ENSMUSG00000029260
Skipped Ugt2b34 ENSMSIG00000003246 No Ensembl homology result for ENSMSIG00000003246
Skipped Ifi27l2a ENSMUSG00000079017 No Ensembl homology result for ENSMUSG00000079017
Skipped Ifi27l2a ENSMSIG00000030944 No Ensembl homology result for ENSMSIG00000030944
Skipped Cyp2f2 ENSNVIG00000012144 No Ensembl homology result for ENSNVIG00000012144
Skipped Lilrb4b ENSMUSG00000112023 No Ensembl homology result for ENSMUSG00000112023
Skipped Bmyc ENSMSIG00000029608 No Ensembl homology result for ENSMSIG00000029608
Skipped Bmyc ENSMUSG0000

,gene_mouse,dataset_support,datasets,mean_log2fc,median_log2fc,mean_abs_log2fc,direction_score,min_qvalue,best_pvalue,contrast_count,consensus_weight,mouse_gene,mouse_ensembl,human_gene,human_ensembl,orthology_type,perc_id,source
0,Npas2,4,"GLDS-245,GLDS-246,GLDS-289,GLDS-48",2.660256,2.505043,2.660256,1.000000,3.433588e-16,3.818704e-20,8,10.641023,Npas2,None,NPAS2,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
2,Slc34a2,3,"GLDS-244,GLDS-245,GLDS-48",3.371168,2.956145,3.371168,1.000000,8.714115e-07,2.979030e-09,8,10.113503,Slc34a2,None,SLC34A2,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
3,Col5a3,3,"GLDS-244,GLDS-245,GLDS-48",2.672380,2.474566,2.672380,1.000000,6.540812e-07,2.032781e-09,4,8.017139,Col5a3,None,COL5A3,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
4,Aqp8,3,"GLDS-244,GLDS-245,GLDS-48",2.638912,1.291768,2.638912,1.000000,3.564103e-07,3.601492e-10,4,7.916736,Aqp8,None,AQP8,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
5,Igkv3-2,3,"GLDS-244,GLDS-289,GLDS-48",1.722424,3.418134,4.285246,0.600000,2.676638e-03,7.871952e-05,10,7.713443,Igkv3-2,None,IGKV@,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
6,Cacna1h,3,"GLDS-244,GLDS-289,GLDS-48",2.489354,2.243153,2.745720,0.857143,4.134197e-06,3.583641e-08,14,7.060424,Cacna1h,None,CACNA1H,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
7,Dmkn,3,"GLDS-244,GLDS-289,GLDS-48",-1.708257,1.729090,11.756778,0.200000,2.421567e-10,4.200443e-12,10,7.054067,Dmkn,None,DMKN,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
8,A2ml1,3,"GLDS-244,GLDS-289,GLDS-48",2.232601,3.048369,2.981950,0.750000,1.953376e-03,2.040085e-05,8,6.709387,A2ml1,None,A2ML1,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
9,Slc30a10,3,"GLDS-244,GLDS-245,GLDS-48",2.222663,1.695842,2.222663,1.000000,9.898681e-09,1.025451e-11,4,6.667990,Slc30a10,None,SLC30A10,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation
10,Ube2u,3,"GLDS-244,GLDS-245,GLDS-48",1.939255,1.412624,1.939255,1.000000,1.440762e-03,2.505290e-04,6,5.817764,Ube2u,None,UBE2U,None,same_symbol_human_validated,NaN,mygene_human_symbol_validation


# CTD Download And Filtering

In [13]:
def pubmed_count(x):
    if x is None or pd.isna(x):
        return 0
    return len([p for p in re.split(r"[|;,]\s*", str(x)) if p.strip()])

def is_bad_chemical(name):
    low = str(name).lower()
    return any(t in low for t in BAD_CHEMICAL_TERMS)

def download_ctd():
    path = CONTENT / "CTD_chem_gene_ixns.csv.gz"

    if path.exists() and path.stat().st_size > 1_000_000:
        print("Using existing CTD:", path)
        return path

    print("Downloading CTD...")
    with requests.get(CTD_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    print("Downloaded:", path, round(path.stat().st_size / 1e6, 2), "MB")
    return path

def load_ctd_edges(human_genes):
    path = download_ctd()

    cols = [
        "ChemicalName", "ChemicalID", "CasRN", "GeneSymbol", "GeneID",
        "GeneForms", "Organism", "OrganismID", "Interaction",
        "InteractionActions", "PubMedIDs",
    ]

    frames = []

    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        reader = pd.read_csv(handle, comment="#", names=cols, chunksize=250_000, low_memory=False)

        for chunk in reader:
            chunk["GeneSymbol"] = chunk.GeneSymbol.astype(str).str.upper()
            chunk["OrganismID"] = pd.to_numeric(chunk.OrganismID, errors="coerce").fillna(-1).astype(int)
            chunk["pubmed_count"] = chunk.PubMedIDs.map(pubmed_count)

            keep = chunk[
                (chunk.OrganismID == 9606)
                & (chunk.GeneSymbol.isin(human_genes))
                & (chunk.pubmed_count >= CONFIG.min_ctd_pubmed_count)
                & (~chunk.ChemicalName.map(is_bad_chemical))
            ].copy()

            if not keep.empty:
                frames.append(keep)

    if not frames:
        return pd.DataFrame(columns=["drug", "gene", "confidence", "source"])

    out = pd.concat(frames, ignore_index=True)
    out = out.rename(columns={"ChemicalName": "drug", "GeneSymbol": "gene"})
    out["drug"] = out.drug.astype(str).str.strip()
    out["confidence"] = 0.80
    out["source"] = "CTD_human_filtered"

    return out[["drug", "gene", "confidence", "source"]].drop_duplicates()

ctd_edges = load_ctd_edges(human_genes)

print("CTD edges:", len(ctd_edges))
print("CTD drugs:", ctd_edges.drug.nunique() if not ctd_edges.empty else 0)
display(ctd_edges.head(20))

Using existing CTD: /content/CTD_chem_gene_ixns.csv.gz
CTD edges: 86
CTD drugs: 11


,drug,gene,confidence,source
0,"7,8-Dihydro-7,8-dihydroxybenzo(a)pyrene 9,10-o...",CDKN1A,0.8,CTD_human_filtered
1,Arsenic Trioxide,ABCB1,0.8,CTD_human_filtered
2,Arsenic Trioxide,CCL2,0.8,CTD_human_filtered
3,Arsenic Trioxide,CDKN1A,0.8,CTD_human_filtered
5,Arsenic Trioxide,DNMT3B,0.8,CTD_human_filtered
6,Arsenic Trioxide,FOS,0.8,CTD_human_filtered
7,Arsenic Trioxide,ITGAM,0.8,CTD_human_filtered
8,Arsenic Trioxide,JUN,0.8,CTD_human_filtered
10,Arsenic Trioxide,ME1,0.8,CTD_human_filtered
11,Arsenic Trioxide,PTGS2,0.8,CTD_human_filtered


# ChEMBL API Validation

In [7]:
def chembl_get(endpoint, params):
    url = f"{CHEMBL_BASE}/{endpoint}.json"
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=40)
            r.raise_for_status()
            return r.json()
        except Exception:
            if attempt == 2:
                raise
            time.sleep(1 + attempt)

def chembl_targets_for_gene(gene):
    data = chembl_get("target/search", {"q": gene, "limit": 20})
    ids = []

    for item in data.get("targets", []):
        organism = str(item.get("organism", "")).lower()
        ttype = str(item.get("target_type", "")).lower()

        if "homo sapiens" not in organism:
            continue
        if not any(k in ttype for k in ["single protein", "protein complex", "protein family"]):
            continue

        tid = item.get("target_chembl_id")
        if tid:
            ids.append(tid)

    return ids

def load_chembl_edges(human_genes):
    rows = []
    genes = sorted(human_genes)[:CONFIG.max_chembl_genes]

    print("Querying ChEMBL genes:", len(genes))

    for i, gene in enumerate(genes, 1):
        try:
            for tid in chembl_targets_for_gene(gene):
                data = chembl_get("mechanism", {"target_chembl_id": tid, "limit": 1000})

                for mech in data.get("mechanisms", []):
                    try:
                        phase = float(mech.get("max_phase") or 0)
                    except Exception:
                        phase = 0

                    if phase < 4:
                        continue

                    drug = mech.get("molecule_pref_name") or mech.get("molecule_chembl_id")
                    if not drug or is_bad_chemical(drug):
                        continue

                    rows.append({
                        "drug": str(drug).strip(),
                        "gene": gene,
                        "confidence": 0.95,
                        "source": "ChEMBL_phase4_api",
                    })

            if i % 25 == 0:
                print(i, "/", len(genes), "genes queried | edges:", len(rows))

            time.sleep(0.08)

        except Exception as e:
            print("ChEMBL skipped", gene, e)

    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=["drug", "gene", "confidence", "source"])

chembl_edges = load_chembl_edges(human_genes)

print("ChEMBL edges:", len(chembl_edges))
print("ChEMBL drugs:", chembl_edges.drug.nunique() if not chembl_edges.empty else 0)
display(chembl_edges.head(20))

Querying ChEMBL genes: 120
25 / 120 genes queried | edges: 6
50 / 120 genes queried | edges: 25
75 / 120 genes queried | edges: 25
100 / 120 genes queried | edges: 25
ChEMBL skipped C2 400 Client Error: Bad Request for url: https://www.ebi.ac.uk/chembl/api/data/target/search.json?q=C2&limit=20
ChEMBL skipped C3 400 Client Error: Bad Request for url: https://www.ebi.ac.uk/chembl/api/data/target/search.json?q=C3&limit=20
ChEMBL edges: 44
ChEMBL drugs: 44


,drug,gene,confidence,source
0,CHEMBL802,ABCC9,0.95,ChEMBL_phase4_api
1,CHEMBL1200338,ABCC9,0.95,ChEMBL_phase4_api
2,CHEMBL2108594,ACVR1,0.95,ChEMBL_phase4_api
3,CHEMBL2109171,ACVR1,0.95,ChEMBL_phase4_api
4,CHEMBL6068336,ACVR1,0.95,ChEMBL_phase4_api
5,CHEMBL5095049,ACVR1,0.95,ChEMBL_phase4_api
6,CHEMBL142635,ADRA1A,0.95,ChEMBL_phase4_api
7,CHEMBL1201535,ADRA1A,0.95,ChEMBL_phase4_api
8,CHEMBL24778,ADRA1A,0.95,ChEMBL_phase4_api
9,CHEMBL1201044,ADRA1A,0.95,ChEMBL_phase4_api


# Combine Drug-Gene Edge

In [8]:
drug_edges = (
    pd.concat([ctd_edges, chembl_edges], ignore_index=True)
    .dropna(subset=["drug", "gene"])
    .drop_duplicates(["drug", "gene"])
)

drug_edges["gene"] = drug_edges.gene.astype(str).str.upper()
drug_edges = drug_edges[drug_edges.gene.isin(human_genes)]

if drug_edges.empty:
    raise ValueError("No drug-gene edges survived CTD/ChEMBL filtering.")

print("Total drug-gene edges:", len(drug_edges))
print("Unique drugs:", drug_edges.drug.nunique())
print("Unique genes targeted:", drug_edges.gene.nunique())
display(drug_edges.head(30))

Total drug-gene edges: 360
Unique drugs: 143
Unique genes targeted: 164


,drug,gene,confidence,source
0,1-Methyl-4-phenylpyridinium,IL1B,0.8,CTD_human_filtered
1,"3,4,5,3',4'-pentachlorobiphenyl",CYP1B1,0.8,CTD_human_filtered
2,"3,4,5,3',4'-pentachlorobiphenyl",IL1B,0.8,CTD_human_filtered
3,"6-(4-chlorophenyl)imidazo(2,1-b)(1,3)thiazole-...",NR1I3,0.8,CTD_human_filtered
4,"6-formylindolo(3,2-b)carbazole",CYP1B1,0.8,CTD_human_filtered
5,8-Bromo Cyclic Adenosine Monophosphate,STAR,0.8,CTD_human_filtered
6,"9,10-Dimethyl-1,2-benzanthracene",CYP1B1,0.8,CTD_human_filtered
7,Acetaminophen,CDKN1A,0.8,CTD_human_filtered
8,Acetylcholine,CHRNA4,0.8,CTD_human_filtered
9,Aflatoxin B1,ADGRG1,0.8,CTD_human_filtered


# Build Heterogeneous Graph

In [9]:
def make_map(values):
    return {v: i for i, v in enumerate(sorted(set(values)))}

gene_map = make_map(consensus_human.human_gene)
drug_map = make_map(drug_edges.drug)

data = HeteroData()
data["gene"].num_nodes = len(gene_map)
data["drug"].num_nodes = len(drug_map)
data["disease"].num_nodes = 1

drug_src = [drug_map[d] for d in drug_edges.drug]
gene_dst = [gene_map[g] for g in drug_edges.gene]

data["drug", "targets", "gene"].edge_index = torch.tensor([drug_src, gene_dst], dtype=torch.long)
data["gene", "rev_targets", "drug"].edge_index = torch.tensor([gene_dst, drug_src], dtype=torch.long)

gene_src = [gene_map[g] for g in consensus_human.human_gene]
weights = consensus_human.consensus_weight.astype(float).to_numpy()
weights = weights / max(float(weights.max()), 1e-8)

data["gene", "causes", "disease"].edge_index = torch.tensor([gene_src, [0] * len(gene_src)], dtype=torch.long)
data["gene", "causes", "disease"].edge_attr = torch.tensor(weights, dtype=torch.float32).view(-1, 1)

print(data)

HeteroData(
  gene={ num_nodes=1122 },
  drug={ num_nodes=143 },
  disease={ num_nodes=1 },
  (drug, targets, gene)={ edge_index=[2, 360] },
  (gene, rev_targets, drug)={ edge_index=[2, 360] },
  (gene, causes, disease)={
    edge_index=[2, 1141],
    edge_attr=[1141, 1],
  }
)


# Define Model

In [10]:
class WeightedGeneDiseaseConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.src = nn.Linear(channels, channels)
        self.dst = nn.Linear(channels, channels)

    def forward(self, src_x, dst_x, edge_index, edge_weight):
        src, dst = edge_index
        msg = self.src(src_x[src]) * edge_weight.view(-1, 1)
        out = torch.zeros_like(dst_x)
        out.index_add_(0, dst, msg)

        denom = torch.zeros(dst_x.size(0), device=dst_x.device)
        denom.index_add_(0, dst, edge_weight.view(-1))

        return out / denom.clamp_min(1e-6).view(-1, 1) + self.dst(dst_x)

class WeightedHeteroGraphSAGE(nn.Module):
    def __init__(self, data, hidden):
        super().__init__()
        self.gene_emb = nn.Embedding(data["gene"].num_nodes, hidden)
        self.drug_emb = nn.Embedding(data["drug"].num_nodes, hidden)
        self.disease_emb = nn.Embedding(data["disease"].num_nodes, hidden)

        self.dg1 = SAGEConv((hidden, hidden), hidden)
        self.gd1 = SAGEConv((hidden, hidden), hidden)
        self.gm1 = WeightedGeneDiseaseConv(hidden)

        self.dg2 = SAGEConv((hidden, hidden), hidden)
        self.gd2 = SAGEConv((hidden, hidden), hidden)
        self.gm2 = WeightedGeneDiseaseConv(hidden)

        self.gn = nn.LayerNorm(hidden)
        self.dn = nn.LayerNorm(hidden)
        self.mn = nn.LayerNorm(hidden)

    def encode(self, edge_index_dict, edge_attr_dict):
        h_gene = self.gene_emb.weight
        h_drug = self.drug_emb.weight
        h_dis = self.disease_emb.weight

        e_dg = edge_index_dict[("drug", "targets", "gene")]
        e_gd = edge_index_dict[("gene", "rev_targets", "drug")]
        e_gm = edge_index_dict[("gene", "causes", "disease")]
        w_gm = edge_attr_dict[("gene", "causes", "disease")].view(-1)

        ng = self.dg1((h_drug, h_gene), e_dg)
        nd = self.gd1((h_gene, h_drug), e_gd)
        nm = self.gm1(h_gene, h_dis, e_gm, w_gm)

        h_gene = F.relu(self.gn(ng))
        h_drug = F.relu(self.dn(nd))
        h_dis = F.relu(self.mn(nm))

        ng = self.dg2((h_drug, h_gene), e_dg)
        nd = self.gd2((h_gene, h_drug), e_gd)
        nm = self.gm2(h_gene, h_dis, e_gm, w_gm)

        h_gene = F.relu(self.gn(ng + h_gene))
        h_drug = F.relu(self.dn(nd + h_drug))
        h_dis = F.relu(self.mn(nm + h_dis))

        return {"gene": h_gene, "drug": h_drug, "disease": h_dis}

    def decode_drug_gene(self, z, edge_index):
        src, dst = edge_index
        return (z["drug"][src] * z["gene"][dst]).sum(dim=-1)

    def decode_drug_disease(self, z):
        return (z["drug"] * z["disease"][0]).sum(dim=-1)

print("Model class ready.")

Model class ready.


# Train Model

In [11]:
def edge_attrs(d):
    return {("gene", "causes", "disease"): d["gene", "causes", "disease"].edge_attr}

def view_with_edges(d, edge_index):
    out = HeteroData()
    out["gene"].num_nodes = d["gene"].num_nodes
    out["drug"].num_nodes = d["drug"].num_nodes
    out["disease"].num_nodes = d["disease"].num_nodes
    out["drug", "targets", "gene"].edge_index = edge_index
    out["gene", "rev_targets", "drug"].edge_index = edge_index.flip(0)
    out["gene", "causes", "disease"].edge_index = d["gene", "causes", "disease"].edge_index
    out["gene", "causes", "disease"].edge_attr = d["gene", "causes", "disease"].edge_attr
    return out

class HardNegativeSampler:
    def __init__(self, edge_index, num_genes):
        self.pos = {(int(edge_index[0, i]), int(edge_index[1, i])) for i in range(edge_index.size(1))}
        self.num_genes = num_genes

    def sample(self, pos_edges):
        neg = []
        for i in range(pos_edges.size(1)):
            drug = int(pos_edges[0, i])
            gene = random.randrange(self.num_genes)
            while (drug, gene) in self.pos:
                gene = random.randrange(self.num_genes)
            neg.append((drug, gene))
        return torch.tensor(neg, dtype=torch.long).t()

edge_index = data["drug", "targets", "gene"].edge_index
perm = torch.randperm(edge_index.size(1))

n_val = max(1, int(edge_index.size(1) * 0.1))
n_test = max(1, int(edge_index.size(1) * 0.1))

val_pos = edge_index[:, perm[:n_val]]
test_pos = edge_index[:, perm[n_val:n_val + n_test]]
train_pos = edge_index[:, perm[n_val + n_test:]]

train_view = view_with_edges(data, train_pos)
sampler = HardNegativeSampler(edge_index, data["gene"].num_nodes)

model = WeightedHeteroGraphSAGE(data, CONFIG.hidden_channels)
opt = torch.optim.Adam(model.parameters(), lr=CONFIG.lr, weight_decay=CONFIG.weight_decay)
loss_fn = nn.MarginRankingLoss(margin=CONFIG.margin)

history = []

for epoch in range(1, CONFIG.epochs + 1):
    model.train()
    opt.zero_grad()

    neg = sampler.sample(train_pos)
    z = model.encode(train_view.edge_index_dict, edge_attrs(train_view))

    pos_score = model.decode_drug_gene(z, train_pos)
    neg_score = model.decode_drug_gene(z, neg)

    loss = loss_fn(pos_score, neg_score, torch.ones_like(pos_score))
    loss.backward()
    opt.step()

    if epoch == 1 or epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            val_neg = sampler.sample(val_pos)
            z = model.encode(train_view.edge_index_dict, edge_attrs(train_view))
            scores = torch.cat([model.decode_drug_gene(z, val_pos), model.decode_drug_gene(z, val_neg)]).numpy()
            labels = np.r_[np.ones(val_pos.size(1)), np.zeros(val_neg.size(1))]
            auc = roc_auc_score(labels, scores)

        history.append({"epoch": epoch, "loss": float(loss), "val_auc": float(auc)})
        print(f"Epoch {epoch:03d} | loss={loss.item():.4f} | val_auc={auc:.4f}")

print("Training complete.")

/tmp/ipykernel_3574/3977805528.py:72: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  history.append({"epoch": epoch, "loss": float(loss), "val_auc": float(auc)})


Epoch 001 | loss=2.7886 | val_auc=0.6026
Epoch 020 | loss=0.0000 | val_auc=0.8002
Epoch 040 | loss=0.0056 | val_auc=0.7770
Epoch 060 | loss=0.0174 | val_auc=0.7801
Epoch 080 | loss=0.0000 | val_auc=0.8086
Epoch 100 | loss=0.0000 | val_auc=0.7832
Epoch 120 | loss=0.0000 | val_auc=0.8063
Training complete.


# Test AUC And Rank Drugs

In [12]:
model.eval()

with torch.no_grad():
    test_neg = sampler.sample(test_pos)
    z = model.encode(train_view.edge_index_dict, edge_attrs(train_view))

    test_scores = torch.cat([
        model.decode_drug_gene(z, test_pos),
        model.decode_drug_gene(z, test_neg),
    ]).numpy()

    test_labels = np.r_[np.ones(test_pos.size(1)), np.zeros(test_neg.size(1))]
    test_auc = roc_auc_score(test_labels, test_scores)

    drug_scores = model.decode_drug_disease(z).numpy()

drug_decoder = {v: k for k, v in drug_map.items()}

ranked = pd.DataFrame([
    {"drug": drug_decoder[i], "score": float(drug_scores[i])}
    for i in range(len(drug_scores))
]).sort_values("score", ascending=False).reset_index(drop=True)

ranked.insert(0, "rank", range(1, len(ranked) + 1))

print("Test AUC:", test_auc)
display(ranked.head(25))

Test AUC: 0.7816358024691359


,rank,drug,score
0,1,CHEMBL2109204,18.363573
1,2,Carnitine,17.733562
2,3,perfluorooctane sulfonic acid,17.101761
3,4,Zoledronic Acid,16.665140
4,5,CHEMBL2108594,16.560656
5,6,N-Formylmethionine Leucyl-Phenylalanine,15.945790
6,7,CHEMBL6068336,15.738514
7,8,Particulate Matter,15.597716
8,9,CHEMBL5095049,15.501101
9,10,perfluorooctanoic acid,15.398432
